# Hyperparameter Tuning

## Objectives
- Load the prepared engineered train, validation, and test splits from `../data/`.
- Tune the baseline tree-based models from notebook 3 with `RandomizedSearchCV`.
- Prefer cuML and GPU-backed estimators where they are available.
- Put extra search budget into the strongest full-feature models, especially XGBoost.
- Evaluate the best searched full-feature models on the held-out test split.


In [ ]:
# Mount Google Drive (only needed in Google Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set the working directory (only needed in Google Colab)
import os
os.chdir('/content/drive/MyDrive/bitcoin-ransomware-classifier/notebooks')
print('Working directory:', os.getcwd())

In [ ]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")
X_val = pd.read_csv("../data/X_val.csv")
X_test = pd.read_csv("../data/X_test.csv")

y_train = pd.read_csv("../data/y_train.csv").squeeze("columns")
y_val = pd.read_csv("../data/y_val.csv").squeeze("columns")
y_test = pd.read_csv("../data/y_test.csv").squeeze("columns")

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier as SkRandomForestClassifier
from sklearn.metrics import average_precision_score, balanced_accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier

GPU_ENABLED = False

try:
    import cudf
    from cuml.ensemble import RandomForestClassifier as cuRandomForestClassifier
    GPU_ENABLED = True
    print("Using cuML RandomForestClassifier on GPU where available.")
except ImportError:
    cudf = None
    cuRandomForestClassifier = None
    print("cuML not available; using sklearn estimators on CPU.")

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as exc:
    XGBClassifier = None
    XGB_AVAILABLE = False
    print(f"XGBoost unavailable in this environment: {exc}")

# cuML takes over the random forest path here so RandomizedSearchCV can try GPU-backed fits when RAPIDS is installed in a Linux/NVIDIA CUDA environment; on macOS this notebook will fall back to CPU instead.
RandomForestClassifier = cuRandomForestClassifier if GPU_ENABLED else SkRandomForestClassifier


In [ ]:
import numpy as np


def score_predictions(y_true, y_pred):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def normalize_predictions(y_pred):
    if hasattr(y_pred, "to_pandas"):
        y_pred = y_pred.to_pandas()
    elif type(y_pred).__module__.startswith("cupy"):
        y_pred = y_pred.get()
    return np.asarray(y_pred)


def run_random_search(model_name, estimator, param_distributions, X_train_split, y_train_split, X_val_split, y_val_split, feature_set, n_iter=12):
    X_search = pd.concat([X_train_split, X_val_split], ignore_index=True)
    y_search = pd.concat([y_train_split, y_val_split], ignore_index=True)
    validation_fold = [-1] * len(X_train_split) + [0] * len(X_val_split)
    splitter = PredefinedSplit(test_fold=validation_fold)

    search = RandomizedSearchCV(
        estimator=clone(estimator),
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="f1",
        cv=splitter,
        random_state=12,
        verbose=1,
        refit=True,
    )
    search.fit(X_search, y_search)

    rows = pd.DataFrame(search.cv_results_)
    rows["model"] = model_name
    rows["feature_set"] = feature_set
    rows["params"] = rows["params"].apply(lambda params: dict(params))

    y_val_pred = normalize_predictions(search.best_estimator_.predict(X_val_split))
    best_summary = {
        "model": model_name,
        "feature_set": feature_set,
        "params": dict(search.best_params_),
        **score_predictions(y_val_split, y_val_pred),
    }

    return rows, best_summary, search.best_estimator_


## Search Space

This section defines the parameter ranges for each model family. The strongest full-feature candidates receive the most search budget, with the widest search assigned to XGBoost.


In [ ]:
search_space = {
    "tree": {
        "estimator": DecisionTreeClassifier(random_state=12),
        "params": {
            "max_depth": [8, 12, 18, None],
            "min_samples_split": [2, 10, 30],
            "min_samples_leaf": [1, 5, 10, 20],
            "class_weight": [None, "balanced"],
        },
        "n_iter": 14,
    },
    "forest": {
        "estimator": RandomForestClassifier(n_estimators=300, random_state=12),
        "params": {
            "n_estimators": [200, 300, 500],
            "max_depth": [12, 24, 36, 48],
            "min_samples_split": [2, 10, 30],
            "min_samples_leaf": [1, 5, 10],
            "max_features": [0.5, 0.75, 1.0],
        },
        "n_iter": 16,
    },
    "grad": {
        "estimator": HistGradientBoostingClassifier(random_state=12),
        "params": {
            "learning_rate": [0.03, 0.05, 0.1],
            "max_depth": [None, 8, 12],
            "max_leaf_nodes": [31, 63, 127],
            "min_samples_leaf": [20, 50, 100],
            "l2_regularization": [0.0, 0.1, 1.0],
        },
        "n_iter": 16,
    },
}

if XGB_AVAILABLE:
    xgb_params = {
        "random_state": 12,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "n_estimators": 400,
    }
    if GPU_ENABLED:
        xgb_params["device"] = "cuda"

    search_space["xgb"] = {
        "estimator": XGBClassifier(**xgb_params),
        "params": {
            "n_estimators": [200, 400, 600],
            "learning_rate": [0.03, 0.05, 0.1],
            "max_depth": [4, 6, 8],
            "min_child_weight": [1, 5, 10],
            "subsample": [0.7, 0.85, 1.0],
            "colsample_bytree": [0.6, 0.8, 1.0],
            "gamma": [0.0, 1.0, 5.0],
            "reg_alpha": [0.0, 0.1, 1.0],
            "reg_lambda": [1.0, 5.0, 10.0],
        },
        "n_iter": 24,
    }

if not GPU_ENABLED:
    search_space["forest"]["params"]["class_weight"] = [None, "balanced"]


## Randomized Search

This section runs `RandomizedSearchCV` for each model on the engineered full-feature dataset. It keeps the validation split fixed so the best settings can be compared consistently across model families.


In [ ]:
search_results = []
best_rows = []
best_models = {}

for model_name, spec in search_space.items():
    print(f"Running RandomizedSearchCV for {model_name} on Full features...")
    result_df, best_summary, best_model = run_random_search(
        model_name=model_name,
        estimator=spec["estimator"],
        param_distributions=spec["params"],
        X_train_split=X_train,
        y_train_split=y_train,
        X_val_split=X_val,
        y_val_split=y_val,
        feature_set="Full",
        n_iter=spec.get("n_iter", 12),
    )
    search_results.append(result_df)
    best_rows.append(best_summary)
    best_models[("Full", model_name)] = best_model

all_results = pd.concat(search_results, ignore_index=True)
all_results[["model", "rank_test_score", "mean_test_score", "params"]].sort_values(["model", "rank_test_score"]).head(10)


## Best Validation Models

This section collects the strongest validation result for each model. It acts as the shortlist for test-set evaluation and later threshold tuning.


In [ ]:
best_validation_models = pd.DataFrame(best_rows).sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)
best_validation_models


## Test Set Evaluation

This section scores the best searched full-feature models on the held-out test split using the default prediction threshold. These results show how the selected model settings generalize before any threshold tuning.


In [ ]:
test_rows = []

for _, row in best_validation_models.iterrows():
    model_name = row["model"]
    params = row["params"]
    model = best_models[("Full", model_name)]

    y_pred = normalize_predictions(model.predict(X_test))

    test_rows.append({
        "model": model_name,
        "feature_set": "Full",
        "params": params,
        **score_predictions(y_test, y_pred),
    })

test_results = pd.DataFrame(test_rows).sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)
test_results


## Threshold Tuning

These cells reuse the already-fitted `best_models` from this session. They tune the decision threshold on validation predictions with a finer sweep, then apply the best threshold to the held-out test split without retraining the models.


In [ ]:
def normalize_scores(values):
    if hasattr(values, "to_pandas"):
        values = values.to_pandas()
    elif type(values).__module__.startswith("cupy"):
        values = values.get()
    values = np.asarray(values)
    if values.ndim == 2 and values.shape[1] > 1:
        values = values[:, 1]
    return values.reshape(-1)


def get_model_scores(model, X):
    if hasattr(model, "predict_proba"):
        return normalize_scores(model.predict_proba(X))
    if hasattr(model, "decision_function"):
        return normalize_scores(model.decision_function(X))
    raise AttributeError(f"{type(model).__name__} does not expose predict_proba or decision_function.")


def find_best_threshold(y_true, scores, thresholds=None):
    thresholds = thresholds if thresholds is not None else np.linspace(0.05, 0.95, 91)
    rows = []

    for threshold in thresholds:
        y_pred = (scores >= threshold).astype(int)
        rows.append({
            "threshold": float(threshold),
            **score_predictions(y_true, y_pred),
        })

    threshold_df = pd.DataFrame(rows).sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)
    return threshold_df, threshold_df.iloc[0].to_dict()


In [ ]:
threshold_search_rows = []
threshold_test_rows = []

for _, row in best_validation_models.iterrows():
    model_name = row["model"]
    params = row["params"]
    model = best_models[("Full", model_name)]

    val_scores = get_model_scores(model, X_val)
    threshold_df, best_threshold_row = find_best_threshold(y_val, val_scores)
    threshold_df["model"] = model_name
    threshold_df["feature_set"] = "Full"
    threshold_df["params"] = str(params)
    threshold_search_rows.append(threshold_df)

    best_threshold = float(best_threshold_row["threshold"])
    test_scores = get_model_scores(model, X_test)
    test_preds = (test_scores >= best_threshold).astype(int)

    threshold_test_rows.append({
        "model": model_name,
        "feature_set": "Full",
        "params": params,
        "threshold": best_threshold,
        **score_predictions(y_test, test_preds),
    })

threshold_search_results = pd.concat(threshold_search_rows, ignore_index=True)
threshold_test_results = pd.DataFrame(threshold_test_rows).sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)

threshold_test_results


In [ ]:
threshold_search_results.to_csv("../data/threshold_search_results.csv", index=False)
threshold_test_results.to_csv("../data/test_results_threshold_tuned_models.csv", index=False)

print("Saved threshold-tuning summaries to ../data/")

In [ ]:
all_results.to_csv("../data/hyperparameter_search_results.csv", index=False)
best_validation_models.to_csv("../data/best_validation_models.csv", index=False)
test_results.to_csv("../data/test_results_tuned_models.csv", index=False)

print("Saved search and test summaries to ../data/")
